# Flight-Delay: Build the star schema

This notebook takes every monthly BTS file that `download.py` saved in `data/raw/` and builds a **star schema** in DuckDB (`data/flights.duckdb`).

## What is a star schema?

A way of laying out tables for analysis. It has two kinds of table:

- A **fact table** with one row per event (here, one row per flight). It holds the numbers we measure (delays, distance) plus **keys**: short codes that point to other tables.
- **Dimension tables** that describe those keys. A fact row only stores `DL`, and `dim_carrier` says `DL` means "Delta Air Lines Inc.". Each dimension has a **primary key (PK)**, a column with a unique value in every row, so each key in the fact table matches exactly one dimension row.

Drawn out, the fact table sits in the middle and the dimensions around it, which is where the "star" name comes from:

```
                 dim_date
                 (date)
                    |
                    | flight_date
                    |
dim_carrier ---- fact_flights ---- dim_airport
(carrier_code)   carrier_code      (airport_code)
                 origin_airport ------^
                 dest_airport   ------^
```

`origin_airport` and `dest_airport` both point to the same `dim_airport` table. One airport list serves both roles.

## Why bother?

Names, cities and calendar facts are stored once instead of being repeated in millions of flight rows. To ask a question like "delay rate on holidays" or "delay rate by state", you join the fact table to the dimension you need.

## Steps

1. Setup: imports and the database connection
2. Find the downloaded monthly CSV files
3. Build `fact_flights` straight from the CSVs
4. Build `dim_date`
5. Build `dim_carrier`
6. Build `dim_airport`
7. Sanity checks

**Before running:** run `python download.py` first. If `03_sql.ipynb` is open with a running kernel, restart or close that kernel first. DuckDB lets only one program write to a database file at a time, so its open connection would block this one.

## 1. Setup

Imports the libraries and opens a connection to `data/flights.duckdb`.

- `duckdb` runs the SQL. It reads the CSV files from disk itself, so the full year of flights (about 7 million rows) never has to be loaded into pandas.
- `pandas` is only used to build the small calendar table (`dim_date`). Its `USFederalHolidayCalendar` already knows the rules for every US federal holiday.
- `Path` from `pathlib` is used to list the files in `data/raw/`.

`data/flights.duckdb` already exists from Phase 1 and holds the `flight_data` table that `03_sql.ipynb` uses. This notebook only adds new tables next to it and does not touch `flight_data`.

In [16]:
from pathlib import Path

import duckdb
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

# Opens the database file (DuckDB creates it if it doesn't exist).
con = duckdb.connect("data/flights.duckdb")

## 2. Find the downloaded files

Lists every monthly CSV in `data/raw/`. Each file name ends in `_<year>_<month>.csv`.

The `*` in the pattern is a wildcard that matches any text, so `On_Time_*.csv` matches every monthly file but not the two lookup tables (`L_UNIQUE_CARRIERS.csv`, `L_AIRPORT.csv`). The same pattern is passed to DuckDB in the next step, so this list is exactly the set of files that will be loaded.

In [17]:
CSV_PATTERN = "data/raw/On_Time_*.csv"

# sorted() puts them in name order. glob() uses the same * wildcard as DuckDB.
monthly_files = sorted(Path("data/raw").glob("On_Time_*.csv"))
for f in monthly_files:
    print(f.name)
print(f"\n{len(monthly_files)} monthly files found")

On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_10.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_11.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_12.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_8.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_9.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_2.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_3.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_4.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_5.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_6.csv
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_7.csv

12 monthly files found


## 3. Build `fact_flights`

One row per flight, for every month at once. DuckDB reads all the CSV files matched by the pattern as if they were one big table.

**Keeping only the needed columns early.** Each BTS file has about 110 columns. The `SELECT` below names just the 10 it needs, and DuckDB skips the rest while it reads, so they never use any memory.

**Reading options:**
- `all_varchar = true` reads every column as plain text. Without it, DuckDB guesses each column's type from a sample of rows, and in Phase 1 pandas found mixed types in some diversion columns. Reading as text and then converting (`CAST`) only the columns we keep means the guessing can't go wrong, and the conversions are written out where you can see them.
- `union_by_name = true` matches columns across files by name instead of by position, in case BTS ever adds or reorders a column in one month.

**The cleaning rules, same as Phase 1 (`02_clean.ipynb`):**
- `dep_hour`: `CRSDepTime` is the scheduled departure as a number like `1430` (2:30 PM). Integer division by 100 (`//`) drops the minutes, so `1430 // 100 = 14`. BTS writes midnight as `2400`, which would give hour 24, so it is mapped to `0`, the same as Phase 1.
- `delayed`: `1` if `ArrDelay` is over 15 minutes, otherwise `0`. If `ArrDelay` is empty (`NULL`), `delayed` is left `NULL` too, since we don't know if that flight was late. Writing `0` there would count those flights as on time.

**Different from Phase 1:** cancelled and diverted flights are **kept**. They have no arrival delay, so their `arr_delay` and `delayed` are `NULL`, and the `cancelled` and `diverted` columns (1 = yes, 0 = no) let each analysis query decide whether to include them.

**Column conversions:**
- `FlightDate` text like `2025-08-01` becomes a real `DATE`.
- The delay and distance columns are stored as text like `-5.00`, so they become `DOUBLE` (a decimal number).
- `Cancelled` and `Diverted` arrive as `1.00` / `0.00`. They are converted to a decimal first and then to a whole number (`INTEGER`), since text `"1.00"` can't go straight to an integer.
- `Reporting_Airline` is the airline code that matches the codes in `L_UNIQUE_CARRIERS`.

`CREATE OR REPLACE TABLE` means re-running this cell rebuilds the table from scratch instead of failing because it already exists.

Running this cell takes a little while, since it reads all twelve files (several GB).

In [18]:
con.execute(f'''
    CREATE OR REPLACE TABLE fact_flights AS
    -- "WITH raw AS (...)" names a temporary result ("raw") that the main SELECT below reads from.
    -- Here "raw" is just the 10 needed columns, still as text, straight from the CSV files.
    WITH raw AS (
        SELECT
            FlightDate, Reporting_Airline, Origin, Dest, CRSDepTime,
            DepDelay, ArrDelay, Cancelled, Diverted, Distance
        FROM read_csv('{CSV_PATTERN}', header = true, all_varchar = true, union_by_name = true)
    )
    SELECT
        -- Keys: these point to the dimension tables
        CAST(FlightDate AS DATE)   AS flight_date,     -- -> dim_date.date
        Reporting_Airline          AS carrier_code,    -- -> dim_carrier.carrier_code
        Origin                     AS origin_airport,  -- -> dim_airport.airport_code
        Dest                       AS dest_airport,    -- -> dim_airport.airport_code

        -- Measures: the facts about each flight
        CASE
            WHEN CAST(CRSDepTime AS INTEGER) // 100 = 24 THEN 0  -- 2400 means midnight -> hour 0
            ELSE CAST(CRSDepTime AS INTEGER) // 100              -- 1430 // 100 = 14
        END                        AS dep_hour,
        CAST(DepDelay AS DOUBLE)   AS dep_delay,       -- minutes; negative = left early
        CAST(ArrDelay AS DOUBLE)   AS arr_delay,       -- minutes; NULL for cancelled/diverted flights
        CASE
            WHEN ArrDelay IS NULL THEN NULL            -- unknown arrival -> unknown delayed
            WHEN CAST(ArrDelay AS DOUBLE) > 15 THEN 1  -- more than 15 minutes late
            ELSE 0
        END                        AS delayed,
        CAST(CAST(Cancelled AS DOUBLE) AS INTEGER) AS cancelled,  -- "1.00" -> 1.0 -> 1
        CAST(CAST(Diverted  AS DOUBLE) AS INTEGER) AS diverted,
        CAST(Distance AS DOUBLE)   AS distance         -- miles
    FROM raw
''')

# fetchone() returns the single result row as a tuple, [0] takes its first value.
print(f"fact_flights rows: {con.execute('SELECT COUNT(*) FROM fact_flights').fetchone()[0]:,}")
con.execute("SELECT * FROM fact_flights LIMIT 5").df()

fact_flights rows: 7,043,858


,flight_date,carrier_code,origin_airport,dest_airport,dep_hour,dep_delay,arr_delay,delayed,cancelled,diverted,distance
0,2025-10-10,AA,SAN,PHX,6,-9.0,-16.0,0,0,0,304.0
1,2025-10-11,AA,SAN,PHX,6,-6.0,-25.0,0,0,0,304.0
2,2025-10-12,AA,SAN,PHX,6,-2.0,1.0,0,0,0,304.0
3,2025-10-13,AA,SAN,PHX,6,-9.0,-20.0,0,0,0,304.0
4,2025-10-14,AA,SAN,PHX,6,-6.0,-12.0,0,0,0,304.0


### Check the column types

`DESCRIBE` lists each column with its type. `flight_date` should be `DATE`, the delays and distance `DOUBLE`, and the codes `VARCHAR` (text).

In [19]:
con.execute("DESCRIBE fact_flights").df()[["column_name", "column_type"]]

,column_name,column_type
0,flight_date,DATE
1,carrier_code,VARCHAR
2,origin_airport,VARCHAR
3,dest_airport,VARCHAR
4,dep_hour,INTEGER
5,dep_delay,DOUBLE
6,arr_delay,DOUBLE
7,delayed,INTEGER
8,cancelled,INTEGER
9,diverted,INTEGER


## 4. Build `dim_date`

One row per calendar day, from the first to the last flight date in `fact_flights`. The **primary key** is `date`, and `fact_flights.flight_date` points to it.

Why a date table, when the date is already in the fact table? It lets an analysis query ask about weekends, seasons or holidays with a simple join, instead of working those out inside every query. Holidays in particular can't be computed with a short formula.

Every day in the range gets a row (`pd.date_range` produces a continuous run of days), so even a day with zero flights would still be there.

**Columns:**
- `day_of_week`: 1 = Monday through 7 = Sunday. This is the same numbering BTS uses in its own `DayOfWeek` column. pandas counts 0 = Monday, so 1 is added.
- `month`: 1 to 12.
- `season`: meteorological seasons, which go by whole months: Winter = Dec, Jan, Feb; Spring = Mar, Apr, May; Summer = Jun, Jul, Aug; Fall = Sep, Oct, Nov.
- `is_weekend`: `True` on Saturday and Sunday.
- `is_holiday`: `True` on US federal holidays, from pandas' `USFederalHolidayCalendar`. When a holiday falls on a weekend, the calendar marks the **observed** weekday instead. For example, July 4, 2026 is a Saturday, so Friday July 3 is marked.

The table is built in pandas because it's small (about 365 rows) and pandas has the holiday calendar. It is then copied into DuckDB.

In [20]:
# Find the first and last flight dates. fetchone() returns them as a (first, last) tuple.
first_day, last_day = con.execute("SELECT MIN(flight_date), MAX(flight_date) FROM fact_flights").fetchone()
print(f"Date range: {first_day} to {last_day}")

# One row per day in that range.
dates = pd.DataFrame({"date": pd.date_range(first_day, last_day, freq="D")})

# .dt gives access to date parts. day_of_week is 0 = Monday ... 6 = Sunday, so add 1.
dates["day_of_week"] = dates["date"].dt.day_of_week + 1
dates["month"] = dates["date"].dt.month

# A dictionary from month number to season name; .map() looks each month up in it.
season_of_month = {12: "Winter", 1: "Winter", 2: "Winter",
                   3: "Spring", 4: "Spring", 5: "Spring",
                   6: "Summer", 7: "Summer", 8: "Summer",
                   9: "Fall", 10: "Fall", 11: "Fall"}
dates["season"] = dates["month"].map(season_of_month)

dates["is_weekend"] = dates["day_of_week"] >= 6  # 6 = Saturday, 7 = Sunday

# holidays() returns every federal holiday date between start and end.
# .isin() then gives True for each row whose date is in that list.
holidays = USFederalHolidayCalendar().holidays(start=first_day, end=last_day)
dates["is_holiday"] = dates["date"].isin(holidays)

print(f"{len(dates)} days, {dates['is_holiday'].sum()} holidays")
dates[dates["is_holiday"]]  # show just the holiday rows

Date range: 2025-08-01 to 2026-07-31
365 days, 11 holidays


,date,day_of_week,month,season,is_weekend,is_holiday
31,2025-09-01,1,9,Fall,False,True
73,2025-10-13,1,10,Fall,False,True
102,2025-11-11,2,11,Fall,False,True
118,2025-11-27,4,11,Fall,False,True
146,2025-12-25,4,12,Winter,False,True
153,2026-01-01,4,1,Winter,False,True
171,2026-01-19,1,1,Winter,False,True
199,2026-02-16,1,2,Winter,False,True
297,2026-05-25,1,5,Spring,False,True
322,2026-06-19,5,6,Summer,False,True


Now copy the DataFrame into DuckDB. `con.register` makes the pandas DataFrame visible to SQL under the name `dates_df`. pandas stores dates as timestamps (date plus a time of 00:00), so `CAST(... AS DATE)` keeps just the date, which matches the `DATE` type of `fact_flights.flight_date` so the two can be joined.

In [21]:
con.register("dates_df", dates)
con.execute('''
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        CAST(date AS DATE) AS date,  -- primary key
        day_of_week,
        month,
        season,
        is_weekend,
        is_holiday
    FROM dates_df
    ORDER BY date
''')
con.unregister("dates_df")  # the DataFrame is no longer needed by SQL

con.execute("SELECT * FROM dim_date LIMIT 5").df()

,date,day_of_week,month,season,is_weekend,is_holiday
0,2025-08-01,5,8,Summer,False,False
1,2025-08-02,6,8,Summer,True,False
2,2025-08-03,7,8,Summer,True,False
3,2025-08-04,1,8,Summer,False,False
4,2025-08-05,2,8,Summer,False,False


## 5. Build `dim_carrier`

One row per airline. The **primary key** is `carrier_code`, and `fact_flights.carrier_code` points to it. The name comes from the BTS lookup table `L_UNIQUE_CARRIERS`, which has two columns: `Code` and `Description` (the airline name).

The lookup table lists about 1,800 carriers from all of BTS history. Only the carriers that actually appear in `fact_flights` are kept, so the dimension is short and every row is relevant. The `WHERE Code IN (SELECT ...)` part does that filtering.

`all_varchar = true` is used again so that codes are always read as text exactly as written (some start with a digit, like `9E`), with no type guessing.

In [22]:
con.execute('''
    CREATE OR REPLACE TABLE dim_carrier AS
    SELECT
        Code        AS carrier_code,  -- primary key
        Description AS carrier_name
    FROM read_csv('data/raw/L_UNIQUE_CARRIERS.csv', header = true, all_varchar = true)
    WHERE Code IN (SELECT DISTINCT carrier_code FROM fact_flights)
    ORDER BY carrier_code
''')

con.execute("SELECT * FROM dim_carrier").df()

,carrier_code,carrier_name
0,AA,American Airlines Inc.
1,AS,Alaska Airlines Inc.
2,B6,JetBlue Airways
3,DL,Delta Air Lines Inc.
4,F9,Frontier Airlines Inc.
5,G4,Allegiant Air
6,HA,Hawaiian Airlines Inc.
7,MQ,Envoy Air
8,NK,Spirit Air Lines
9,OH,PSA Airlines Inc.


## 6. Build `dim_airport`

One row per airport. The **primary key** is `airport_code`. Both `fact_flights.origin_airport` and `fact_flights.dest_airport` point to it.

The data comes from the BTS lookup table `L_AIRPORT`. Its `Description` packs three things into one text value, in the shape `City, ST: Airport Name`:

```
"New York, NY: John F. Kennedy International"
 ^^^^^^^^  ^^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 city      state   airport name
```

The parsing works in two cuts:
1. Split at the **first** `": "`. The left side is `City, ST`, the right side is the airport name.
2. Split the left side at the **last** `", "`. Before it is the city, after it is the state. The last comma is used in case a city name itself ever contains a comma.

This is done with `regexp_extract`, which pulls out the part of a text that matches a pattern (a **regular expression**). In the patterns:
- `^` means the start of the text, `$` the end.
- `[^:]*` means "any number of characters that are not a colon".
- `.*` means "anything" (any number of any characters).
- `( ... )` marks the part to pull out. The number after the pattern (`1` or `2`) says which bracketed group to return.

**Edge cases** (checked while writing this notebook):
- 2 of the about 6,900 rows don't have the `City, ST: Name` shape (`LEU` and `ZZZ`, "Unknown Point in Alaska"). The patterns don't match those, `regexp_extract` returns an empty text, and `NULLIF(..., '')` turns that into `NULL`. For those rows the whole description is used as the airport name, and city and state are left `NULL`.
- For airports outside the US, the part after the comma is a country (for example `"Lima, Peru: ..."`), so `state` holds `Peru`. US territories use BTS's own codes (Puerto Rico is `PR`, Guam is `TT`).

Like `dim_carrier`, only airports that appear in `fact_flights` (as an origin **or** a destination) are kept. `UNION` stacks the two code lists and removes duplicates.

In [23]:
con.execute('''
    CREATE OR REPLACE TABLE dim_airport AS
    WITH lookup AS (
        SELECT
            Code AS airport_code,
            Description,
            -- Everything before the first ": " (e.g. "New York, NY")
            NULLIF(regexp_extract(Description, '^([^:]*): (.*)$', 1), '') AS city_state,
            -- Everything after the first ": " (e.g. "John F. Kennedy International")
            NULLIF(regexp_extract(Description, '^([^:]*): (.*)$', 2), '') AS name_part
        FROM read_csv('data/raw/L_AIRPORT.csv', header = true, all_varchar = true)
    )
    SELECT
        airport_code,                                                    -- primary key
        NULLIF(regexp_extract(city_state, '^(.*), ([^,]*)$', 1), '') AS city,   -- before the last ", "
        NULLIF(regexp_extract(city_state, '^(.*), ([^,]*)$', 2), '') AS state,  -- after the last ", "
        COALESCE(name_part, Description) AS airport_name  -- COALESCE = first non-NULL value
    FROM lookup
    WHERE airport_code IN (
        SELECT origin_airport FROM fact_flights
        UNION
        SELECT dest_airport FROM fact_flights
    )
    ORDER BY airport_code
''')

print(f"dim_airport rows: {con.execute('SELECT COUNT(*) FROM dim_airport').fetchone()[0]}")
con.execute("SELECT * FROM dim_airport LIMIT 10").df()

dim_airport rows: 359


,airport_code,city,state,airport_name
0,ABE,Allentown/Bethlehem/Easton,PA,Lehigh Valley International
1,ABI,Abilene,TX,Abilene Regional
2,ABQ,Albuquerque,NM,Albuquerque International Sunport
3,ABR,Aberdeen,SD,Aberdeen Regional
4,ABY,Albany,GA,Southwest Georgia Regional
5,ACK,Nantucket,MA,Nantucket Memorial
6,ACT,Waco,TX,Waco Regional
7,ACV,Arcata/Eureka,CA,California Redwood Coast Humboldt County
8,ACY,Atlantic City,NJ,Atlantic City International
9,ADK,Adak Island,AK,Adak


### Check the parsing

Any airport that ended up with a missing city or state is listed here, so parsing problems are easy to spot. An empty result means every airport used in the flights was parsed into all three parts.

In [24]:
con.execute("SELECT * FROM dim_airport WHERE city IS NULL OR state IS NULL").df()

,airport_code,city,state,airport_name


## 7. Sanity checks

Three checks that the schema is built correctly. Each prints a table, and the `assert` lines stop the notebook with an error message if a check fails (the same idea as the `assert` in `03_sql.ipynb`).

### 7a. Row counts per table

How many rows each table has. Rough expectations: `fact_flights` in the millions (about 600,000 flights a month), `dim_date` about 365 (one year of days), `dim_carrier` a couple of dozen airlines, `dim_airport` a few hundred airports.

In [25]:
row_counts = con.execute('''
    SELECT 'fact_flights' AS table_name, COUNT(*) AS num_rows FROM fact_flights
    UNION ALL SELECT 'dim_date',    COUNT(*) FROM dim_date
    UNION ALL SELECT 'dim_carrier', COUNT(*) FROM dim_carrier
    UNION ALL SELECT 'dim_airport', COUNT(*) FROM dim_airport
''').df()
row_counts

,table_name,num_rows
0,fact_flights,7043858
1,dim_date,365
2,dim_carrier,14
3,dim_airport,359


### 7b. Primary key uniqueness

A primary key must be unique and never empty, or one fact row could match two dimension rows and get counted twice in a join.

For each dimension, this compares `COUNT(*)` (all rows) with `COUNT(DISTINCT key)` (number of different key values), and counts `NULL` keys. A good result: `num_rows` equals `distinct_keys`, and `null_keys` is 0.

**Why check instead of declaring `PRIMARY KEY` on the table?** In DuckDB, declaring a `PRIMARY KEY` (or `FOREIGN KEY`) also builds an index behind the scenes. Indexes are out of scope for this step, so the rule is verified with a query instead.

In [26]:
pk_checks = con.execute('''
    SELECT 'dim_date' AS table_name, 'date' AS pk_column,
           COUNT(*) AS num_rows, COUNT(DISTINCT date) AS distinct_keys,
           COUNT(*) - COUNT(date) AS null_keys   -- COUNT(column) skips NULLs, so the difference = number of NULLs
    FROM dim_date
    UNION ALL
    SELECT 'dim_carrier', 'carrier_code',
           COUNT(*), COUNT(DISTINCT carrier_code), COUNT(*) - COUNT(carrier_code)
    FROM dim_carrier
    UNION ALL
    SELECT 'dim_airport', 'airport_code',
           COUNT(*), COUNT(DISTINCT airport_code), COUNT(*) - COUNT(airport_code)
    FROM dim_airport
''').df()
display(pk_checks)

assert (pk_checks["num_rows"] == pk_checks["distinct_keys"]).all(), "A primary key has duplicate values"
assert (pk_checks["null_keys"] == 0).all(), "A primary key has NULL values"
print("All primary keys are unique and non-NULL.")

,table_name,pk_column,num_rows,distinct_keys,null_keys
0,dim_date,date,365,365,0
1,dim_carrier,carrier_code,14,14,0
2,dim_airport,airport_code,359,359,0


All primary keys are unique and non-NULL.


### 7c. Fact rows with no matching dimension row

Every key in `fact_flights` should find its row in the matching dimension. A fact row whose key has no match is called an **orphan**. It would silently drop out of any query that joins to that dimension.

How the query finds orphans: a `LEFT JOIN` keeps **every** fact row, and fills the dimension's columns with `NULL` when no match is found. So `WHERE <dimension key> IS NULL` keeps exactly the fact rows that had no match. A `NULL` key in the fact table also counts as an orphan, since `NULL` never matches anything.

`dim_airport` is checked twice, once for `origin_airport` and once for `dest_airport`. A good result is 0 for all four.

In [27]:
orphans = con.execute('''
    SELECT 'flight_date -> dim_date' AS fact_key, COUNT(*) AS orphan_rows
    FROM fact_flights f LEFT JOIN dim_date d ON f.flight_date = d.date
    WHERE d.date IS NULL
    UNION ALL
    SELECT 'carrier_code -> dim_carrier', COUNT(*)
    FROM fact_flights f LEFT JOIN dim_carrier c ON f.carrier_code = c.carrier_code
    WHERE c.carrier_code IS NULL
    UNION ALL
    SELECT 'origin_airport -> dim_airport', COUNT(*)
    FROM fact_flights f LEFT JOIN dim_airport a ON f.origin_airport = a.airport_code
    WHERE a.airport_code IS NULL
    UNION ALL
    SELECT 'dest_airport -> dim_airport', COUNT(*)
    FROM fact_flights f LEFT JOIN dim_airport a ON f.dest_airport = a.airport_code
    WHERE a.airport_code IS NULL
''').df()
display(orphans)

assert (orphans["orphan_rows"] == 0).all(), "Some fact rows have keys with no matching dimension row"
print("Every fact row matches a row in each dimension.")

,fact_key,orphan_rows
0,flight_date -> dim_date,0
1,carrier_code -> dim_carrier,0
2,origin_airport -> dim_airport,0
3,dest_airport -> dim_airport,0


Every fact row matches a row in each dimension.


## Done

Close the connection so other notebooks (like `03_sql.ipynb`) can open `data/flights.duckdb`. The four tables are saved in the file and will be there next time it is opened.

In [28]:
con.close()